## 📦 SETUP & IMPORTS

In [0]:
%pip install simple-salesforce pyyaml
dbutils.library.restartPython()

In [0]:
%run ./salesforce

In [0]:
import datetime
import traceback
import pandas
import warnings
import re
import json

from datetime import timezone
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit
from delta.tables import DeltaTable

warnings.filterwarnings("ignore", category=FutureWarning)
spark = SparkSession.builder.getOrCreate()

## 🛤️ CONFIGURATION & PATHS

In [0]:
STATE_PATH = "/Workspace/Users/e713362@edp.pt/sf_databricks_pbi/data/logs/state.json"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

## 🧹 DATA PREPARATION

In [0]:
def sanitizar_dataframe(df):
    df = df.copy()

    # Replace invalid Delta characters with underscore
    df.columns = [
        re.sub(r"[ ,;{}()\n\t=&]", "_", str(col)).strip("_")
        for col in df.columns
    ]

    # Avoid duplicate column names after sanitization
    seen = {}
    new_cols = []

    for col in df.columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")

    df.columns = new_cols

    for col in df.columns:
        # Flatten dicts/lists
        if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
            df[col] = df[col].apply(
                lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
            )

        # Convert mixed-type columns to string
        if df[col].apply(type).nunique() > 2:
            df[col] = df[col].astype(str).replace("None", None).replace("nan", None)

    return df

### 🥉 BRONZE

In [0]:
def bronze(df_pandas, nome_tabela):
    if df_pandas is None or df_pandas.empty:
        print(f"bronze.sf_{nome_tabela}: sem dados.")
        return

    df_pandas = sanitizar_dataframe(df_pandas)
    df_spark = spark.createDataFrame(df_pandas)
    df_spark = df_spark \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source_system", lit("Salesforce"))

    tabela = f"{BRONZE_SCHEMA}.sf_{nome_tabela}"

    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(tabela)

    print(f"'{tabela}' guardada ({df_pandas.shape[0]} registos)")

### 🥈 SILVER

In [0]:
def silver(tabela_origem, tabela_destino, chave=None, manter_ingestion_timestamp=False):
    origem = f"{BRONZE_SCHEMA}.{tabela_origem}"
    destino = f"{SILVER_SCHEMA}.{tabela_destino}"

    if not spark.catalog.tableExists(origem):
        print(f"{origem}: tabela não existe.")
        return

    df = spark.table(origem)

    # Remove Bronze-only metadata
    if "_source_system" in df.columns:
        df = df.drop("_source_system")

    if not manter_ingestion_timestamp and "_ingestion_timestamp" in df.columns:
        df = df.drop("_ingestion_timestamp")

    # Cast date/timestamp-like columns
    for c in df.columns:
        c_lower = c.lower()

        if ("date" in c_lower or c_lower.startswith("data_") or "_data_" in c_lower or c_lower.endswith("_timestamp")):
            df = df.withColumn(c, F.to_timestamp(F.col(c)))

    # Safer numeric casts
    for c in df.columns:
        c_lower = c.lower()

        numeric_like = (
            "cost" in c_lower
            or "custo" in c_lower
            or "price" in c_lower
            or "valor" in c_lower
            or "value" in c_lower
            or "quantity" in c_lower
            or "duration" in c_lower
            or "days" in c_lower
            or "power" in c_lower
            or "potencia" in c_lower
            or "latitude" in c_lower
            or "longitude" in c_lower
            or c_lower.startswith("number_of")
            or "_number_of_" in c_lower
            or c_lower.startswith("n_")
        )

        string_identifier = (
            "id" in c_lower
            or "name" in c_lower
            or "casenumber" in c_lower
            or "card_number" in c_lower
            or "serial_number" in c_lower
            or "phone" in c_lower
            or "email" in c_lower
            or "postal" in c_lower
            or "zip" in c_lower
            or "cpe" in c_lower
            or "nipc" in c_lower
            or "country" in c_lower
            or "currency" in c_lower
            or "contract" in c_lower
            or "contrato" in c_lower
            or "quote" in c_lower
            or "productcode" in c_lower
        )

        if numeric_like and not string_identifier:
            df = df.withColumn(c, F.expr(f"try_cast(`{c}` as double)"))

    # Deduplicate
    if chave:
        if isinstance(chave, list):
            chave_existente = [c for c in chave if c in df.columns]
            if chave_existente:
                df = df.dropDuplicates(chave_existente)
            else:
                df = df.dropDuplicates()
        else:
            if chave in df.columns:
                df = df.dropDuplicates([chave])
            else:
                df = df.dropDuplicates()
    else:
        df = df.dropDuplicates()

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"{destino} criada/atualizada com {df.count()} registos")

### 🥇 GOLD

In [0]:
def gold(tabela_origem, tabela_destino, colunas=None, renomear=None):
    origem = f"{SILVER_SCHEMA}.{tabela_origem}"
    destino = f"{GOLD_SCHEMA}.{tabela_destino}"

    if not spark.catalog.tableExists(origem):
        print(f"{origem}: tabela não existe.")
        return

    df = spark.table(origem)

    if colunas:
        colunas_existentes = [c for c in colunas if c in df.columns]
        missing = [c for c in colunas if c not in df.columns]

        if missing:
            print(f"⚠ {destino}: colunas inexistentes ignoradas: {missing}")

        df = df.select(*colunas_existentes)

    if renomear:
        for old, new in renomear.items():
            if old in df.columns:
                df = df.withColumnRenamed(old, new)

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"{destino} criada/atualizada com {df.count()} registos")

## ⚡ PROCESSING & UPSERT

In [0]:
def ler_state():
    try:
        with open(STATE_PATH, "r") as f:
            state = json.load(f)
        return state.get("last_sync")
    except FileNotFoundError:
        return None

def guardar_state():
    state = {"last_sync": datetime.datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000+0000")}
    with open(STATE_PATH, "w") as f:
        json.dump(state, f)
    print(f"✔ State guardado: {state['last_sync']}")

In [0]:
def upsert_bronze(df_pandas, nome_tabela, chave):
    if df_pandas is None or df_pandas.empty:
        print(f"⚠ bronze.sf_{nome_tabela}: sem dados.")
        return

    df_pandas = sanitizar_dataframe(df_pandas)
    df_spark = spark.createDataFrame(df_pandas)
    df_spark = df_spark \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source_system", lit("Salesforce"))

    tabela_completa = f"{BRONZE_SCHEMA}.sf_{nome_tabela}"

    if spark.catalog.tableExists(tabela_completa):
        delta_table = DeltaTable.forName(spark, tabela_completa)

        if isinstance(chave, list):
            condicao = " AND ".join([f"target.{c} = source.{c}" for c in chave])
        else:
            condicao = f"target.{chave} = source.{chave}"

        delta_table.alias("target") \
            .merge(df_spark.alias("source"), condicao) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()

        print(f"✔ '{tabela_completa}' atualizada (merge) com {df_pandas.shape[0]} registos.")
    else:
        df_spark.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(tabela_completa)

        print(f"✔ '{tabela_completa}' criada com {df_pandas.shape[0]} registos.")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

print("✔ Schemas created: bronze, silver, gold")

## 🚀 MAIN EXECUTION

In [0]:
def main():
    try:
        print("Conectando ao Salesforce...")
        sf = ligacao_salesforce()
        if sf is None:
            print("Erro na ligação.")
            return
        print("Ligação estabelecida com sucesso.")

        lista_ids = ids_ativos(sf)
        if not lista_ids:
            print("Sem ativos.")
            return
        print(f"Número de ativos: {len(lista_ids)}")

        last_sync = ler_state()
        if last_sync:
            print(f"Última sincronização: {last_sync}")
        else:
            print("Primeira execução — carga completa.")

        periodo = [datetime.date.today() - datetime.timedelta(days=3000), datetime.date.today()]

        ### BRONZE LAYER

        # Ativos
        ativos = obter_ativos(sf_=sf, lista=lista_ids)
        bronze(ativos, "ativos")

        # Decide if Eventos / Pedidos need full load or incremental load
        eventos_table_exists = spark.catalog.tableExists("bronze.sf_eventos")
        po_table_exists = spark.catalog.tableExists("bronze.sf_pedidos_operacao")

        if eventos_table_exists and last_sync:
            desde_eventos = last_sync
            print("Eventos: carga incremental.")
        else:
            desde_eventos = None
            print("Eventos: carga completa.")

        if po_table_exists and last_sync:
            desde_po = last_sync
            print("Pedidos de Operação: carga incremental.")
        else:
            desde_po = None
            print("Pedidos de Operação: carga completa.")

        # Eventos
        eventos = obter_eventos(
            sf_=sf,
            lista=lista_ids,
            periodo=periodo,
            meio=[],
            estado=[],
            desde=desde_eventos
        )

        # Pedidos de Operação
        pedidos_operacao = obter_po(
            sf_=sf,
            lista=lista_ids,
            meio=[],
            estado=[],
            desde=desde_po
        )

        # Task Owner 
        eventos = construir_task_owner(eventos, pedidos_operacao)

        # Upsert Bronze
        upsert_bronze(eventos, "eventos", chave="case_id")
        upsert_bronze(pedidos_operacao, "pedidos_operacao", chave="case_id")

        # Empresa
        empresa = obter_empresa(sf_=sf, lista=lista_ids)
        bronze(empresa, "empresa")

        # Obra
        obra = obter_obra(sf_=sf, lista_id=lista_ids)
        bronze(obra, "obra")

        # CT + Equipamentos + Tomadas
        ct, eq, tomadas = obter_ct_eq_tomadas(sf_=sf, lista=lista_ids)
        bronze(ct, "ct_me")
        bronze(eq, "equipamentos_me")
        bronze(tomadas, "tomadas_me")

        # Sintomas dos Eventos Pai
        if eventos is not None and not eventos.empty and "case_id" in eventos.columns:
            lista_cases = eventos["case_id"].dropna().astype(str).unique().tolist()
        elif eventos is not None and not eventos.empty and "Id" in eventos.columns:
            lista_cases = eventos["Id"].dropna().astype(str).unique().tolist()
        else:
            lista_cases = []

        if lista_cases:
            sintomas_eventos_pai = obter_sintoma_eventos_pai(sf_=sf, lista_solicitacoes=lista_cases)
        else:
            sintomas_eventos_pai = pandas.DataFrame()

        if sintomas_eventos_pai is not None and not sintomas_eventos_pai.empty:
            if "incident_reason_name" not in sintomas_eventos_pai.columns:
                sintomas_eventos_pai["incident_reason_name"] = ""

        upsert_bronze(sintomas_eventos_pai, "sintomas_eventos_pai", chave="case_id")

        ### SILVER LAYER

        silver("sf_ativos", "ativos", chave="salesforce_id")
        silver("sf_eventos", "eventos", chave="case_id")
        silver("sf_pedidos_operacao", "pedidos_operacao", chave="case_id")
        silver("sf_empresa", "empresa", chave=["salesforce_id", "empresa_id"])
        silver("sf_obra", "obra", chave=["salesforce_id", "obra_id"])
        silver("sf_ct_me", "ct_me", chave="ct_id")
        silver("sf_equipamentos_me", "equipamentos_me", chave="eq_id")
        silver("sf_tomadas_me", "tomadas_me", chave="componente_eq_id")
        silver("sf_sintomas_eventos_pai", "sintomas_eventos_pai", chave="case_id")

        ### GOLD LAYER

        gold("ativos", "ativos")
        gold("eventos", "eventos")
        gold("pedidos_operacao", "pedidos_operacao")
        gold("empresa", "empresa")
        gold("obra", "obra")
        gold("ct_me", "ct_me")
        gold("equipamentos_me", "equipamentos_me")
        gold("tomadas_me", "tomadas_me")
        gold("sintomas_eventos_pai", "sintomas_eventos_pai")

        guardar_state()
        print("Pipeline concluído com sucesso!")

    except Exception as e:
        print("Erro durante execução:")
        print(e)
        traceback.print_exc()

In [0]:
main()

## ✅ VERIFICATION

In [0]:
schemas = {
    "🥉 BRONZE": {
        "schema": "bronze",
        "tabelas": [
            "sf_ativos", "sf_eventos", "sf_pedidos_operacao",
            "sf_empresa", "sf_obra", "sf_ct_me",
            "sf_equipamentos_me", "sf_tomadas_me", "sf_sintomas_eventos_pai"
        ]
    },
    "🥈 SILVER": {
        "schema": "silver",
        "tabelas": [
            "ativos", "eventos", "pedidos_operacao",
            "empresa", "obra", "ct_me",
            "equipamentos_me", "tomadas_me", "sintomas_eventos_pai"
        ]
    },
    "🥇 GOLD": {
        "schema": "gold",
        "tabelas": [
            "ativos", "eventos", "pedidos_operacao",
            "empresa", "obra", "ct_me",
            "equipamentos_me", "tomadas_me", "sintomas_eventos_pai"
        ]
    }
}

for layer_name, config in schemas.items():
    schema = config["schema"]
    tabelas = config["tabelas"]

    print(f"\n{'='*60}")
    print(f"  {layer_name}")
    print(f"{'='*60}")

    for tabela in tabelas:
        full_name = f"{schema}.{tabela}"
        try:
            count = spark.sql(f"SELECT COUNT(*) as n FROM {full_name}").collect()[0]["n"]
            print(f"  ✔ {full_name:45s} {count:>8} registos")
        except Exception as e:
            print(f"  ❌ {full_name:45s} ERRO: {e}")

In [0]:
tables = [
    "sf_ativos",
    "sf_eventos",
    "sf_pedidos_operacao",
    "sf_empresa",
    "sf_obra",
    "sf_ct_me",
    "sf_equipamentos_me",
    "sf_tomadas_me",
    "sf_sintomas_eventos_pai"
]

for t in tables:
    print(f"\n{'='*80}")
    print(f"BRONZE TABLE: bronze.{t}")
    print(f"{'='*80}")
    spark.sql(f"DESCRIBE bronze.{t}").show(200, truncate=False)

In [0]:
tables = [
    "sf_ativos",
    "sf_eventos",
    "sf_pedidos_operacao",
    "sf_empresa",
    "sf_obra",
    "sf_ct_me",
    "sf_equipamentos_me",
    "sf_tomadas_me",
    "sf_sintomas_eventos_pai"
]

for t in tables:
    print(f"\n{'='*80}")
    print(f"SAMPLE: bronze.{t}")
    print(f"{'='*80}")
    spark.sql(f"SELECT * FROM bronze.{t} LIMIT 5").show(truncate=False)
